This is the file where the structure of the HRM should be defined. This includes the training loop and the __init__ and forward of the network. It should not include the actual High and Low Level architectures, which are encoder-only transformers.

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import RandAugment
import transformers

from transformers import get_cosine_schedule_with_warmup


import random

from tqdm import tqdm
import sys


from google.colab import drive

torch.manual_seed(0)

drive.mount('/content/drive', force_remount=True)


base_dir = "/content/drive/MyDrive/Junior/Second Semester/CS 4782 - DL/Final Project - HRM"
sys.path.append(base_dir)
from Model.HRM_Model import HRM
from Model.HRM_Components import Encoder, HighLevel, LowLevel, Head
from Datasets.Sudoku_DataLoader import get_loaders

sys.path.append(base_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("meta")
print(F"Device set to {device}")

Mounted at /content/drive
Device set to cuda


In [7]:
train_size = 1000
test_size = 200
batch_size = 64
train_dataloader, test_dataloader = get_loaders(train_size, test_size, batch_size)


In [8]:
#Define Model Hyperparameters
d_model = 512
M = 1
N = 2
T = 2
n_layers = 4
n_heads = 8
vocab_size = 10
lr = 1e-4
beta1 = .9
beta2 = .95
weight_decay = .1
num_epochs = 10
num_warmup_steps = 2000

high_level = HighLevel(d_model=d_model, n_layers=n_layers, n_heads=n_heads, intermediate_size=4*d_model)
low_level = HighLevel(d_model=d_model, n_layers=n_layers, n_heads=n_heads, intermediate_size=4*d_model)
head = Head(vocab_size=vocab_size, d_model=d_model)
encoder = Encoder(vocab_size=vocab_size, d_model=d_model, max_len=81)


HRM_model = HRM(low_level, high_level, encoder, head, M, N, T).to(device)

# Define the optimizer
optimizer = optim.AdamW(
    HRM_model.parameters(),
    lr=lr,
    betas=(beta1, beta2),
    weight_decay=weight_decay
)
num_training_steps = len(train_dataloader) * num_epochs
num_warmup_steps = num_warmup_steps

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
    num_cycles=0.5  # standard cosine
)

In [9]:
def train_hrm_deepsup(model, loader, optimizer, loss_fn, scheduler=None, num_epochs=10, vocab_size=10):
    """
    Train HRM with deep supervision for multiple epochs and report stats.

    model      : HRM model
    loader     : DataLoader with (input_seq, target_seq)
    optimizer  : optimizer
    loss_fn    : loss function (nn.CrossEntropyLoss)
    scheduler  : optional learning rate scheduler
    num_epochs : number of epochs to train
    vocab_size : size of discrete token vocabulary
    """
    print(f"Number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        correct_tokens = 0
        total_tokens = 0

        for x, y in tqdm(loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            x = x.to(device)
            y = y.to(device)
            x_embed = model.encode(x)  # trainable encoder
            B, L, d = x_embed.shape
            z_H = torch.zeros(B, L, d, device=x.device)
            z_L = torch.zeros(B, L, d, device=x.device)

            for m in range(model.M):
                # Detach x_embed after first segment so encoder only trained once
                x_in = x_embed if m == 0 else x_embed.detach()
                z_H, z_L = model.step(z_H, z_L, x_in)

                # Segment-wise forward + loss
                # y_pred = model.head(z_H)               # (B, L, vocab_size)
                # y_pred_flat = y_pred.view(B * L, vocab_size)
                # y_flat = y.view(-1)                     # (B*L,)

                # y_train = y.clone()
                # y_train[x != 0] = -100                 # ignore givens, train only on blanks
                # y_train_flat = y_train.view(-1)

                # loss = loss_fn(y_pred_flat, y_train_flat)


                # Segment-wise forward
                y_pred = model.head(z_H)  # (B, L, vocab_size)

                pred = y_pred.view(B * L, vocab_size)
                x_flat = x.view(-1)
                y_flat = y.view(-1)

                blank_mask = (x_flat == 0)
                given_mask = (x_flat != 0)

                # loss on blanks (solve Sudoku)
                loss_blank = loss_fn(pred[blank_mask], y_flat[blank_mask])

                # loss on givens (identity / copy constraint)
                loss_given = loss_fn(pred[given_mask], x_flat[given_mask])

                loss = loss_blank + loss_given


                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()

                # Detach hidden states for next segment
                z_H = z_H.detach()
                z_L = z_L.detach()

                total_loss += loss.item()

                # Compute token-level accuracy

                mask = (x.view(-1) == 0)

                y_pred_labels = pred.argmax(dim=1)

                correct_tokens += (y_pred_labels[mask] == y_flat[mask]).sum().item()
                total_tokens += mask.sum().item()


                # y_pred_labels = y_pred_flat.argmax(dim=1)
                # correct_tokens += (y_pred_labels == y_flat).sum().item()
                # total_tokens += y_flat.numel()

        avg_loss = total_loss / (len(loader) * model.M)
        acc = correct_tokens / total_tokens

        print(f"Epoch {epoch+1}: Avg Loss = {avg_loss:.4f}, Token Accuracy = {acc*100:.2f}%")

    return model
trained_model = train_hrm_deepsup(HRM_model,train_dataloader, optimizer,nn.CrossEntropyLoss(ignore_index=-100), scheduler, num_epochs=num_epochs, vocab_size=vocab_size)
trained_model.eval()

Number of trainable parameters: 25,270,794


Epoch 1/10: 100%|██████████| 40/40 [01:22<00:00,  2.07s/it]


Epoch 1: Avg Loss = 4.5518, Token Accuracy = 11.02%


Epoch 2/10: 100%|██████████| 40/40 [01:21<00:00,  2.05s/it]


Epoch 2: Avg Loss = 3.8084, Token Accuracy = 11.26%


...

Epoch 9/10: 100%|██████████| 40/40 [01:22<00:00,  2.06s/it]


Epoch 9: Avg Loss = 2.1854, Token Accuracy = 13.71%


Epoch 10/10: 100%|██████████| 40/40 [01:22<00:00,  2.05s/it]

Epoch 10: Avg Loss = 2.1842, Token Accuracy = 13.65%


HRM(
  (low_level): HighLevel(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (high_level): HighLevel(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizable

In [10]:
#Visualize a sudoku
data_iter = iter(test_dataloader)
x_batch, y_batch = next(data_iter)
x = x_batch[0]
y = y_batch[0]
# x,y = train_dataset[random.randint(0,train_size-1)]
x = torch.tensor(x, dtype=torch.long)
print("X: " + str(x))
print("Prediction: " + str(trained_model.predict(x)))
print("Y: " + str(y))


X: tensor([0, 9, 0, 0, 0, 1, 2, 0, 0, 0, 3, 0, 0, 2, 8, 4, 0, 6, 0, 6, 0, 0, 0, 0,
        0, 8, 0, 0, 7, 0, 0, 0, 0, 1, 4, 0, 0, 2, 0, 0, 5, 0, 7, 0, 0, 0, 0, 3,
        0, 0, 0, 0, 0, 2, 0, 1, 0, 9, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 5, 0, 0, 0,
        0, 8, 7, 2, 0, 6, 0, 0, 4])
Prediction: tensor([[5, 9, 5, 5, 5, 1, 2, 5, 5, 5, 3, 5, 5, 2, 8, 4, 5, 6, 5, 6, 5, 5, 5, 5,
         5, 8, 5, 5, 7, 5, 5, 5, 5, 1, 4, 5, 5, 2, 5, 5, 5, 5, 7, 5, 5, 5, 5, 3,
         5, 5, 5, 5, 5, 2, 5, 1, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5, 7, 5, 5, 5, 5, 5,
         5, 8, 7, 2, 5, 6, 5, 5, 4]], device='cuda:0')
Y: tensor([5, 9, 8, 4, 6, 1, 2, 7, 3, 7, 3, 1, 5, 2, 8, 4, 9, 6, 2, 6, 4, 3, 9, 7,
        5, 8, 1, 8, 7, 9, 6, 3, 2, 1, 4, 5, 4, 2, 6, 1, 5, 9, 7, 3, 8, 1, 5, 3,
        8, 7, 4, 9, 6, 2, 6, 1, 5, 9, 4, 3, 8, 2, 7, 3, 4, 2, 7, 8, 5, 6, 1, 9,
        9, 8, 7, 2, 1, 6, 3, 5, 4])


/tmp/ipykernel_37644/1069738194.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.long)
